In [1]:
"""
Password Strength Analyzer (Thiranex Task 1)
- Checks length, complexity, entropy, weak patterns, uniqueness
- Suggests stronger alternatives (secrets module)
- Optional SQLite DB of salted PBKDF2 hashes to prevent password reuse
"""

import getpass
import hashlib
import hmac
import math
import os
import re
import secrets
import sqlite3
import string
import sys

DB_FILE = "password_history.db"
PBKDF2_ITERATIONS = 200_000
MIN_LENGTH = 12

COMMON_PASSWORDS = {
    "123456", "123456789", "12345678", "12345", "1234567", "1234567890",
    "password", "password1", "password123", "passw0rd", "p@ssw0rd",
    "qwerty", "qwerty123", "qwertyuiop", "abc123", "abcd1234", "111111",
    "000000", "123123", "iloveyou", "admin", "admin123", "welcome",
    "welcome1", "login", "letmein", "monkey", "dragon", "football",
    "baseball", "master", "sunshine", "princess", "shadow", "superman",
    "batman", "trustno1", "whatever", "freedom", "hello123", "india123",
    "kolkata", "cricket", "krishna", "ganesh", "mahadev", "test1234",
    "changeme", "default", "guest", "root", "user", "secret", "asdfgh",
    "zxcvbn", "1q2w3e4r", "1qaz2wsx", "qazwsx", "q1w2e3r4", "iloveu",
}

KEYBOARD_ROWS = ["qwertyuiop", "asdfghjkl", "zxcvbnm", "1234567890"]

LEET_MAP = {"a": "@", "e": "3", "i": "!", "o": "0", "s": "$", "t": "7", "l": "1"}

WORDS = [
    "river", "mango", "cloud", "tiger", "lotus", "monsoon", "silver", "ember",
    "falcon", "harbor", "jungle", "meadow", "orbit", "pepper", "quartz",
    "rocket", "saffron", "thunder", "velvet", "willow", "zephyr", "anchor",
    "bamboo", "candle", "desert", "forest", "glacier", "island", "jasmine",
    "kettle", "lantern", "marble", "nectar", "oyster", "pencil", "rabbit",
    "spiral", "temple", "umbrella", "voyage", "walnut", "yonder", "bridge",
    "copper", "dragon", "eclipse", "feather", "granite", "horizon", "indigo",
    "journey", "kingdom", "lagoon", "mirror", "nebula", "orchid", "puzzle",
    "ripple", "sunrise", "trident", "utopia", "vortex", "whisper", "xenon",
    "yellow", "zigzag",
]

SYMBOLS = "!@#$%^&*()-_=+[]{};:,.?"


# ---------------------------- Analysis ----------------------------
def character_pool_size(password):
    pool = 0
    if re.search(r"[a-z]", password):
        pool += 26
    if re.search(r"[A-Z]", password):
        pool += 26
    if re.search(r"\d", password):
        pool += 10
    if re.search(r"[^A-Za-z0-9]", password):
        pool += 32
    return pool


def calculate_entropy(password):
    """Entropy in bits = length * log2(pool size)."""
    pool = character_pool_size(password)
    if pool == 0:
        return 0.0
    return len(password) * math.log2(pool)


def has_sequence(password, run=3):
    lowered = password.lower()
    sources = [string.ascii_lowercase, string.digits] + KEYBOARD_ROWS
    for src in sources:
        for s in (src, src[::-1]):
            for i in range(len(s) - run + 1):
                if s[i:i + run] in lowered:
                    return True
    return False


def has_repeats(password, run=3):
    return re.search(r"(.)\1{%d,}" % (run - 1), password) is not None


def normalize_leet(password):
    reverse = {"@": "a", "3": "e", "!": "i", "1": "l", "0": "o",
               "$": "s", "7": "t", "5": "s", "4": "a"}
    return "".join(reverse.get(c, c) for c in password.lower())


def is_common(password):
    lowered = password.lower()
    if lowered in COMMON_PASSWORDS or normalize_leet(password) in COMMON_PASSWORDS:
        return True
    stem = re.sub(r"[\d\W_]+$", "", lowered)
    if len(stem) >= 4 and (stem in COMMON_PASSWORDS or normalize_leet(stem) in COMMON_PASSWORDS):
        return True
    return False


def analyze_password(password):
    findings = []
    length = len(password)

    if length >= 16:
        findings.append((True, f"Length is excellent ({length} characters)."))
    elif length >= MIN_LENGTH:
        findings.append((True, f"Length is good ({length} characters)."))
    elif length >= 8:
        findings.append((False, f"Length is short ({length}). Use at least {MIN_LENGTH} characters."))
    else:
        findings.append((False, f"Length is too short ({length}). Use at least {MIN_LENGTH} characters."))

    checks = {
        "lowercase letters": bool(re.search(r"[a-z]", password)),
        "uppercase letters": bool(re.search(r"[A-Z]", password)),
        "digits": bool(re.search(r"\d", password)),
        "special characters": bool(re.search(r"[^A-Za-z0-9]", password)),
    }
    for name, ok in checks.items():
        findings.append((ok, f"Contains {name}." if ok else f"Missing {name}."))
    classes = sum(checks.values())

    common = is_common(password)
    findings.append((not common, "Not found in the common-password list."
                     if not common else "This is a very common password (or a trivial variation)."))
    seq = has_sequence(password)
    findings.append((not seq, "No obvious sequences (abc, 123, qwe)."
                     if not seq else "Contains an easy sequence such as abc, 123 or qwe."))
    rep = has_repeats(password)
    findings.append((not rep, "No repeated characters in a row."
                     if not rep else "Contains repeated characters such as 'aaa'."))

    unique_ratio = len(set(password)) / length if length else 0
    if length and unique_ratio < 0.5:
        findings.append((False, "Many characters are reused; use more variety."))

    entropy = calculate_entropy(password)
    score = min(100.0, entropy / 80.0 * 100.0)
    if length < 8:
        score = min(score, 30)
    if classes < 3:
        score -= 10
    if seq:
        score -= 15
    if rep:
        score -= 10
    if unique_ratio < 0.5:
        score -= 10
    if common:
        score = min(score, 5)
    score = int(max(0, min(100, score)))

    if score < 30:
        rating = "VERY WEAK"
    elif score < 50:
        rating = "WEAK"
    elif score < 70:
        rating = "MODERATE"
    elif score < 85:
        rating = "STRONG"
    else:
        rating = "VERY STRONG"

    return {"score": score, "rating": rating, "entropy": entropy,
            "length": length, "findings": findings}


# --------------------------- Suggestions ---------------------------
def generate_random_password(length=16):
    alphabet = string.ascii_letters + string.digits + SYMBOLS
    while True:
        pwd = "".join(secrets.choice(alphabet) for _ in range(length))
        if (re.search(r"[a-z]", pwd) and re.search(r"[A-Z]", pwd)
                and re.search(r"\d", pwd) and re.search(r"[^A-Za-z0-9]", pwd)
                and not has_sequence(pwd) and not has_repeats(pwd)):
            return pwd


def generate_passphrase(words=4):
    chosen = [secrets.choice(WORDS).capitalize() for _ in range(words)]
    return "-".join(chosen) + f"-{secrets.randbelow(90) + 10}{secrets.choice('!@#$%')}"


def harden_password(password):
    base = "".join(
        LEET_MAP.get(c.lower(), c) if (c.lower() in LEET_MAP and secrets.randbelow(2)) else c
        for c in password
    )
    if base:
        i = secrets.randbelow(len(base))
        base = base[:i] + base[i].upper() + base[i + 1:]
    suffix = "".join(secrets.choice(string.digits + SYMBOLS) for _ in range(4))
    result = base + suffix
    while len(result) < 14:
        result += secrets.choice(string.ascii_letters + string.digits + SYMBOLS)
    return result


def suggest_alternatives(password):
    """Always returns suggestions that score STRONG or better (verified)."""
    hardened = None
    for _ in range(50):
        candidate = harden_password(password)
        if analyze_password(candidate)["score"] >= 70:
            hardened = candidate
            break
    if hardened is None:
        hardened = generate_random_password(14)
        label = "Fresh password (yours is too predictable)"
    else:
        label = "Hardened version of yours"
    return [
        (label, hardened),
        ("Random password (16 chars)", generate_random_password(16)),
        ("Memorable passphrase", generate_passphrase(4)),
    ]


# ------------------- Optional: password history DB -------------------
def hash_password(password, salt=None):
    """Salted PBKDF2-HMAC-SHA256. Returns (salt_hex, hash_hex)."""
    if salt is None:
        salt = os.urandom(16)
    digest = hashlib.pbkdf2_hmac("sha256", password.encode("utf-8"), salt, PBKDF2_ITERATIONS)
    return salt.hex(), digest.hex()


class PasswordHistory:
    """Stores only salted hashes, never plain-text passwords."""

    def __init__(self, path=DB_FILE):
        self.conn = sqlite3.connect(path)
        self.conn.execute(
            """CREATE TABLE IF NOT EXISTS password_history (
                   id INTEGER PRIMARY KEY AUTOINCREMENT,
                   username TEXT NOT NULL,
                   salt TEXT NOT NULL,
                   pw_hash TEXT NOT NULL,
                   created_at TEXT DEFAULT CURRENT_TIMESTAMP)"""
        )
        self.conn.commit()

    def was_used(self, username, password):
        rows = self.conn.execute(
            "SELECT salt, pw_hash FROM password_history WHERE username = ?", (username,)
        ).fetchall()
        for salt_hex, stored in rows:
            _, candidate = hash_password(password, bytes.fromhex(salt_hex))
            if hmac.compare_digest(candidate, stored):
                return True
        return False

    def save(self, username, password):
        salt_hex, digest = hash_password(password)
        self.conn.execute(
            "INSERT INTO password_history (username, salt, pw_hash) VALUES (?, ?, ?)",
            (username, salt_hex, digest),
        )
        self.conn.commit()

    def count(self, username):
        return self.conn.execute(
            "SELECT COUNT(*) FROM password_history WHERE username = ?", (username,)
        ).fetchone()[0]

    def close(self):
        self.conn.close()


# ------------------------------ CLI ------------------------------
def bar(score, width=30):
    filled = int(width * score / 100)
    return "[" + "#" * filled + "-" * (width - filled) + f"] {score}/100"


def print_report(result):
    print("\n" + "=" * 56)
    print(f" Strength : {result['rating']}")
    print(f" Score    : {bar(result['score'])}")
    print(f" Entropy  : {result['entropy']:.1f} bits")
    print("=" * 56)
    for ok, msg in result["findings"]:
        print(f"  [{'PASS' if ok else 'FAIL'}] {msg}")


def print_suggestions(password):
    print("\nSuggested stronger alternatives:")
    for label, pwd in suggest_alternatives(password):
        r = analyze_password(pwd)
        print(f"  - {label:<28} {pwd}   ({r['rating']}, {r['entropy']:.0f} bits)")


def read_password(prompt="Enter password: "):
    try:
        return getpass.getpass(prompt)
    except (EOFError, KeyboardInterrupt):
        print()
        return ""


def analyze_flow(history):
    password = read_password()
    if not password:
        print("No password entered.")
        return
    result = analyze_password(password)
    print_report(result)

    username = input("\nUsername (press Enter to skip reuse check): ").strip()
    if username:
        if history.was_used(username, password):
            print("  [FAIL] You have used this password before. Choose a new one.")
            result["score"] = 0
        else:
            print(f"  [PASS] Not reused ({history.count(username)} previous password(s) on record).")
            if result["score"] >= 50:
                if input("Save this password hash to history? (y/n): ").strip().lower() == "y":
                    history.save(username, password)
                    print("  Saved securely (salted PBKDF2 hash, no plain text).")

    if result["score"] < 85:
        print_suggestions(password)


def main():
    history = PasswordHistory()
    menu = """
=========== PASSWORD STRENGTH ANALYZER ===========
 1. Analyze a password
 2. Generate a strong password
 3. Generate a passphrase
 4. Show how a password is hashed (demo)
 5. Exit
"""
    try:
        while True:
            print(menu)
            choice = input("Choose an option (1-5): ").strip()
            if choice == "1":
                analyze_flow(history)
            elif choice == "2":
                n = input("Length (default 16): ").strip()
                length = int(n) if n.isdigit() and int(n) >= 8 else 16
                pwd = generate_random_password(length)
                print(f"\n  {pwd}")
                print_report(analyze_password(pwd))
            elif choice == "3":
                pwd = generate_passphrase()
                print(f"\n  {pwd}")
                print_report(analyze_password(pwd))
            elif choice == "4":
                pwd = read_password("Enter a password to hash: ")
                salt, digest = hash_password(pwd)
                print(f"\n  Salt (random)   : {salt}")
                print(f"  PBKDF2-SHA256   : {digest}")
                print("  Same password + new salt gives a different hash every time:")
                print(f"  Second hash     : {hash_password(pwd)[1]}")
            elif choice == "5":
                print("Goodbye! Stay secure.")
                break
            else:
                print("Invalid choice, enter 1-5.")
    finally:
        history.close()

In [2]:
samples = ["123456", "Password@123", "saheli2003", "Tr0ub4dor&3xyz",
           "Saffron-Orbit-Lotus-Tiger-47!"]
print(f"{'Password':32} {'Rating':12} {'Score':>5} {'Entropy':>8}")
print("-" * 62)
for pw in samples:
    r = analyze_password(pw)
    print(f"{pw:32} {r['rating']:12} {r['score']:>5} {r['entropy']:>7.0f}b")

print("\nDetailed report for 'Password@123':")
print_report(analyze_password("Password@123"))
print_suggestions("Password@123")

Password                         Rating       Score  Entropy
--------------------------------------------------------------
123456                           VERY WEAK        0      20b
Password@123                     VERY WEAK        5      79b
saheli2003                       MODERATE        54      52b
Tr0ub4dor&3xyz                   VERY STRONG     85      92b
Saffron-Orbit-Lotus-Tiger-47!    VERY STRONG    100     190b

Detailed report for 'Password@123':

 Strength : VERY WEAK
 Score    : [#-----------------------------] 5/100
 Entropy  : 78.7 bits
  [PASS] Length is good (12 characters).
  [PASS] Contains lowercase letters.
  [PASS] Contains uppercase letters.
  [PASS] Contains digits.
  [PASS] Contains special characters.
  [FAIL] This is a very common password (or a trivial variation).
  [FAIL] Contains an easy sequence such as abc, 123 or qwe.
  [PASS] No repeated characters in a row.

Suggested stronger alternatives:
  - Fresh password (yours is too predictable) GD5i^r9?X{N

In [3]:
h = PasswordHistory("demo_history.db")
h.save("saheli", "MyOldPass#2026xyz")
print("Old password reused? ->", h.was_used("saheli", "MyOldPass#2026xyz"))   # True
print("New password reused? ->", h.was_used("saheli", "BrandNew$Pass9876"))   # False

salt, digest = hash_password("MyOldPass#2026xyz")
print("\nSalt   :", salt)
print("PBKDF2 :", digest)
print("Same password, new salt -> different hash:", hash_password("MyOldPass#2026xyz")[1])
h.close()

Old password reused? -> True
New password reused? -> False

Salt   : 8f03691f3e6a3c64fa1f88c4b000b639
PBKDF2 : b7cdffa0d47442bbbef44f0ccee233bcd7bdf7ce0a6bb014561c204bb14d42b6
Same password, new salt -> different hash: 06dbd91c9ac09af05776e7e9d862781a83fd8638787905f55749d532ebb6a621


In [4]:
main()



=========== PASSWORD STRENGTH ANALYZER ===========
 1. Analyze a password
 2. Generate a strong password
 3. Generate a passphrase
 4. Show how a password is hashed (demo)
 5. Exit

Choose an option (1-5): 1
Enter password: ··········

 Strength : VERY STRONG
 Score    : [###########################---] 90/100
 Entropy  : 72.1 bits
  [FAIL] Length is short (11). Use at least 12 characters.
  [PASS] Contains lowercase letters.
  [PASS] Contains uppercase letters.
  [PASS] Contains digits.
  [PASS] Contains special characters.
  [PASS] Not found in the common-password list.
  [PASS] No obvious sequences (abc, 123, qwe).
  [PASS] No repeated characters in a row.

Username (press Enter to skip reuse check): Saheli Sarkar
  [PASS] Not reused (0 previous password(s) on record).
Save this password hash to history? (y/n): y
  Saved securely (salted PBKDF2 hash, no plain text).

=========== PASSWORD STRENGTH ANALYZER ===========
 1. Analyze a password
 2. Generate a strong password
 3. Generat